## 1.       ABSA

In [8]:
# Prerequis : uv add transformers torch datasets sentencepiece scikit-learn
# Le Bloc 1 tourne sans reseau. Les Blocs 3+ necessitent un acces
# reseau a huggingface.co (bloque dans le sandbox utilise pour ecrire
# ce code, a lancer chez toi)

# --- BLOC 1 : extraction d'aspects, avec le filtre corrige ---
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from aspect_sentiment.absa import extract_aspect_candidates

avis_test = [
    "The delivery was slow but the product quality is excellent",
    "Customer service was rude and the refund took forever",
    "Great packaging, arrived in perfect condition, fast shipping",
]
for avis in avis_test:
    print(f"Avis : {avis}")
    print(f"  Aspects : {extract_aspect_candidates(avis)}\n")

# verification du filtre (bug reel : "time"/"improvement" extraits a tort)
avis_piege = (
    "The delivery was fast this time but customer service still "
    "needs improvement, though the price is fair"
)
aspects_piege = extract_aspect_candidates(avis_piege)
print(f"Avis piege : {avis_piege}")
print(f"  Aspects (filtre) : {aspects_piege}")
assert "time" not in aspects_piege
assert "improvement" not in aspects_piege
print("  OK : plus de faux positifs 'time'/'improvement'")

Avis : The delivery was slow but the product quality is excellent
  Aspects : ['delivery', 'product quality']

Avis : Customer service was rude and the refund took forever
  Aspects : ['Customer service', 'refund']

Avis : Great packaging, arrived in perfect condition, fast shipping
  Aspects : ['Great packaging', 'condition', 'shipping']

Avis piege : The delivery was fast this time but customer service still needs improvement, though the price is fair
  Aspects (filtre) : ['delivery', 'customer service', 'price']
  OK : plus de faux positifs 'time'/'improvement'


In [9]:
# --- BLOC 2 : charger le vrai dataset SemEval (anglais) ---
from aspect_sentiment.absa import load_semeval_absa

(train_textes, train_aspects, train_labels, eval_textes, eval_aspects, eval_labels) = (
    load_semeval_absa()
)

print(f"\nTrain : {len(train_textes)} exemples, Eval : {len(eval_textes)}")

Splits disponibles : ['train', 'test']
Utilisation des splits natifs train/test du dataset
Colonnes : ['text', 'span', 'label', 'ordinal']  |  3693 lignes train, 1134 lignes test
ATTENTION [test] : 0 exemple extrait sur 1134 lignes brutes. Raisons de rejet : {'text': 0, 'aspect': 0, 'label': 1134}
  Valeurs brutes de label/polarity rencontrees : {''}
  Exemple de ligne brute : {'text': 'The bread is top notch as well.', 'span': 'bread', 'label': '', 'ordinal': 0}
ATTENTION : le split 'test' officiel n'a aucune etiquette de polarite exploitable -- repli sur un decoupage manuel du split 'train' a la place.
2954 exemples train, 739 exemples eval

Train : 2954 exemples, Eval : 739


In [10]:
# --- BLOC 3 : entrainer le modele ABSA sur l'anglais (DistilBERT) ---
from aspect_sentiment.absa import load_absa_classifier, train_absa_model
from transformers_arch.fine_tuning import evaluate_fine_tuned_model

model, tokenizer = load_absa_classifier(num_labels=3)
print("Parametres du modele ABSA :", sum(p.numel() for p in model.parameters()))

trainer = train_absa_model(
    model,
    tokenizer,
    train_textes,
    train_aspects,
    train_labels,
    eval_textes,
    eval_aspects,
    eval_labels,
    epochs=6,
)
resultats = evaluate_fine_tuned_model(trainer)
print("\nResultats d'evaluation (anglais) :", resultats)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Parametres du modele ABSA : 66955779


/home/ensai/Documents/Repositories_Git/sentiment-analysis-platform/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.657095,0.726658,0.599061
2,No log,0.563002,0.765900,0.694986
3,0.574281,0.559954,0.778078,0.705453
4,0.574281,0.598218,0.786198,0.729470
5,0.574281,0.618133,0.791610,0.727266
6,0.261702,0.654092,0.788904,0.724504


...

/home/ensai/Documents/Repositories_Git/sentiment-analysis-platform/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/ensai/Documents/Repositories_Git/sentiment-analysis-platform/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.261702,0.559954,6,0.778078,0.705453



Resultats d'evaluation (anglais) : {'eval_loss': 0.5599536299705505, 'eval_accuracy': 0.7780784844384303, 'eval_f1': 0.7054531903666107}


In [16]:
# --- BLOC 6 : publier le modele "phare" du projet sur Hugging Face ---
# necessite un compte + token (voir docs/huggingface_guide.md)
# decommente pour publier reellement (choisis le modele EN ou XLM-R
# multilingue selon ce que tu veux mettre en avant) :

from transformers_arch.fine_tuning import push_to_hub

push_to_hub(trainer.model, tokenizer, "Steeve2ml/globatrend-absa-english-classifier")
print("\nPour publier : decommente les lignes push_to_hub ci-dessus.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.



Pour publier : decommente les lignes push_to_hub ci-dessus.


In [1]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Steeve2ml/globatrend-sentiment-distilbert")
model = AutoModelForSequenceClassification.from_pretrained(
    "Steeve2ml/globatrend-absa-english-classifier", device_map="auto"
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
# --- BLOC 4 : prediction complete sur un NOUVEL avis multi-aspects ---
from aspect_sentiment.absa import predict_aspect_sentiment

# IMPORTANT : trainer.model (pas 'model'), garanti a jour meme avec
# load_best_model_at_end=True (voir Phase 6 pour le detail du piege)
modele_entraine = trainer.model

nouvel_avis = "Fast shipping, great product, but the website was confusing\
     and the warranty terms are unclear."
aspects_detectes = extract_aspect_candidates(nouvel_avis)
print(f"\nNouvel avis : {nouvel_avis}")
print(f"Aspects detectes automatiquement : {aspects_detectes}")

resultats_finaux = predict_aspect_sentiment(
    nouvel_avis, aspects_detectes, modele_entraine, tokenizer
)
print("\n=== SORTIE FINALE ABSA (aspect -> sentiment) ===")
for aspect, sentiment in resultats_finaux.items():
    print(f"  {aspect:20} -> {sentiment}")

# A retenir : c'est EXACTEMENT la sortie que la problematique du
# projet demande -- pas un sentiment global unique, mais un sentiment
# PAR ASPECT, extrait et classifie automatiquement de bout en bout.


Nouvel avis : Fast shipping, great product, but the website was confusing and the warranty terms are unclear.
Aspects detectes automatiquement : ['Fast shipping', 'product', 'website', 'warranty terms']

=== SORTIE FINALE ABSA (aspect -> sentiment) ===
  Fast shipping        -> positive
  product              -> positive
  website              -> negative
  warranty terms       -> negative


In [ ]:
# --- BLOC 4 : prediction complete sur un NOUVEL avis multi-aspects ---
from aspect_sentiment.absa import predict_aspect_sentiment

# IMPORTANT : trainer.model (pas 'model'), garanti a jour meme avec
# load_best_model_at_end=True (voir Phase 6 pour le detail du piege)
modele_entraine = trainer.model

nouvel_avis = (
    "The delivery was good this time but customer service still needs improvement"
)
aspects_detectes = extract_aspect_candidates(nouvel_avis)
print(f"\nNouvel avis : {nouvel_avis}")
print(f"Aspects detectes automatiquement : {aspects_detectes}")

resultats_finaux = predict_aspect_sentiment(
    nouvel_avis, aspects_detectes, modele_entraine, tokenizer
)
print("\n=== SORTIE FINALE ABSA (aspect -> sentiment) ===")
for aspect, sentiment in resultats_finaux.items():
    print(f"  {aspect:20} -> {sentiment}")

# A retenir : c'est EXACTEMENT la sortie que la problematique du
# projet demande -- pas un sentiment global unique, mais un sentiment
# PAR ASPECT, extrait et classifie automatiquement de bout en bout.


Nouvel avis : The delivery was good this time but customer service still needs improvement
Aspects detectes automatiquement : ['delivery', 'customer service']

=== SORTIE FINALE ABSA (aspect -> sentiment) ===
  delivery             -> positive
  customer service     -> negative


In [34]:
# Teste chaque clause ISOLEMENT, comme si c'etait 4 avis separes
phrases_isolees = {
    "the delivery was fast",
    "the product quality is excellent",
    "the customer service was rude",
    "the refund process took forever",
}
for phrase in phrases_isolees:
    aspects = extract_aspect_candidates(phrase)
    resultat = predict_aspect_sentiment(phrase, aspects, modele_entraine, tokenizer)
    print(f"{phrase!r:45} -> {resultat}")

'the product quality is excellent'            -> {'product quality': 'positive'}
'the delivery was fast'                       -> {'delivery': 'positive'}
'the customer service was rude'               -> {'customer service': 'negative'}
'the refund process took forever'             -> {'refund process': 'negative'}


In [10]:
# --- BLOC 5 : ABSA MULTILINGUE -- transfert ZERO-SHOT (le vrai scope Phase 9) ---
# CORRECTION IMPORTANTE : le modele multilingue doit etre XLM-R, PAS
# DistilBERT (anglais seulement) -- DistilBERT ne permettrait
# structurellement aucun transfert vers d'autres langues. On repart
# donc d'un NOUVEAU modele XLM-R, entraine UNE SEULE FOIS sur
# l'anglais (memes donnees SemEval que Bloc 2/3), puis evalue en
# zero-shot sur ES/DE/FR/HI -- aucune donnee d'entrainement etiquetee
# necessaire dans ces langues.
from aspect_sentiment.absa import (
    evaluate_multilingual_zero_shot,
    load_multilingual_absa_classifier,
)

model_xlmr, tokenizer_xlmr = load_multilingual_absa_classifier(num_labels=3)
print("\nParametres XLM-R :", sum(p.numel() for p in model_xlmr.parameters()))

trainer_xlmr = train_absa_model(
    model_xlmr,
    tokenizer_xlmr,
    train_textes,
    train_aspects,
    train_labels,
    eval_textes,
    eval_aspects,
    eval_labels,
    epochs=8,
)

resultats_multilingues = evaluate_multilingual_zero_shot(trainer_xlmr, tokenizer_xlmr)

print("\n{:8} {:>10} {:>10}".format("Langue", "Accuracy", "F1"))
for lang, res in resultats_multilingues.items():
    print(
        f"{lang:8} {res.get('eval_accuracy', float('nan')):>10.3f} "
        f"{res.get('eval_f1', float('nan')):>10.3f}"
    )

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Parametres XLM-R : 278045955


/home/ensai/Documents/Repositories_Git/sentiment-analysis-platform/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.717232,0.722598,0.516816
2,No log,0.650171,0.764547,0.664202
3,0.733968,0.625561,0.774019,0.693031
4,0.733968,0.590610,0.791610,0.724939
5,0.733968,0.609562,0.807848,0.742244
6,0.418677,0.648868,0.813261,0.747508
7,0.418677,0.677853,0.815968,0.748777
8,0.418677,0.755116,0.810555,0.746694


...

/home/ensai/Documents/Repositories_Git/sentiment-analysis-platform/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.418677,0.640533,8,0.666667,0.555556


/home/ensai/Documents/Repositories_Git/sentiment-analysis-platform/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.418677,1.250270,8,0.500000,0.333333



Langue     Accuracy         F1
es            0.667      0.667
de            0.667      0.667
fr            0.667      0.556
hi            0.500      0.333


In [ ]:
# --- BLOC 4 : prediction complete sur un NOUVEL avis multi-aspects ---
from aspect_sentiment.absa import predict_aspect_sentiment

# IMPORTANT : trainer.model (pas 'model'), garanti a jour meme avec
# load_best_model_at_end=True (voir Phase 6 pour le detail du piege)
modele_entraine = trainer_xlmr.model
tokenizer = tokenizer_xlmr

nouvel_avis = "la livraison était rapide cette fois mais le service \
    client doit encore s'améliorer"
# nouvel_avis_traduit = translate_to_english(nouvel_avis, source_lang="fr")
nouvel_avis_traduit = nouvel_avis  # deja en anglais pour ce test
aspects_detectes = extract_aspect_candidates(nouvel_avis_traduit)
print(f"\nNouvel avis : {nouvel_avis}")
print(f"Aspects detectes automatiquement : {aspects_detectes}")

resultats_finaux = predict_aspect_sentiment(
    nouvel_avis_traduit, aspects_detectes, modele_entraine, tokenizer
)
print("\n=== SORTIE FINALE ABSA (aspect -> sentiment) ===")
for aspect, sentiment in resultats_finaux.items():
    print(f"  {aspect:20} -> {sentiment}")

# A retenir : c'est EXACTEMENT la sortie que la problematique du
# projet demande -- pas un sentiment global unique, mais un sentiment
# PAR ASPECT, extrait et classifie automatiquement de bout en bout.


Nouvel avis : la livraison était rapide cette fois mais le service client doit encore s'améliorer
Aspects detectes automatiquement : ['la livraison était rapide cette fois mais le service client', "s'améliorer"]

=== SORTIE FINALE ABSA (aspect -> sentiment) ===
  la livraison était rapide cette fois mais le service client -> positive
  s'améliorer          -> neutral


In [41]:
# --- BLOC 6 : publier le modele "phare" du projet sur Hugging Face ---
# necessite un compte + token (voir docs/huggingface_guide.md)
# decommente pour publier reellement (choisis le modele EN ou XLM-R
# multilingue selon ce que tu veux mettre en avant) :

from transformers_arch.fine_tuning import push_to_hub

push_to_hub(
    trainer_xlmr.model, tokenizer_xlmr, "Steeve2ml/globatrend-absa-xlm-classifier"
)
print("\nPour publier : decommente les lignes push_to_hub ci-dessus.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Pour publier : decommente les lignes push_to_hub ci-dessus.
